
# Agentic Tool Workflows with Gemini

Exploring manual function calling and agent loops using
Gemini, Python, and the Open-Meteo weather API.

Adapted from the Udemy course examples:
- [Tool Calling](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/tool_calling.ipynb)
- [Building Agents from Scratch](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/learning_agents_from_scratch.ipynb)

The original OpenAI examples are adapted to Gemini,
with additional safeguards for API usage and execution.


In [5]:

%pip install -q -U openai requests

from openai import OpenAI
import getpass
import json
import requests

MODEL = "gemini-3.8-flash"

gemini_key = getpass.getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


Enter your Gemini API key: ··········



## 1. Manual Function Calling

The model requests a function call using structured arguments.
Python executes the function, then sends its result back to
the model.

Unlike automatic function calling, the application controls
each step.


In [6]:

def get_weather(latitude, longitude):
    """Retrieve current temperature in Celsius."""

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m"
    }

    response = requests.get(
        url,
        params=params,
        timeout=15
    )

    response.raise_for_status()
    data = response.json()

    return data["current"]["temperature_2m"]


In [7]:

temperature = get_weather(48.8566, 2.3522)

print(f"Temperature in Paris: {temperature}°C")


Temperature in Paris: 16.6°C


In [8]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current temperature in Celsius for a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "Latitude of the location"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "Longitude of the location"
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    }
]


In [9]:

messages = [
    {
        "role": "user",
        "content": (
            "What's the current temperature in Paris? "
            "Use coordinates 48.8566, 2.3522."
        )
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print("Tool calls:", message.tool_calls)
print("Text response:", message.content)


Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_2237768', function=Function(arguments='{"latitude":48.8566,"longitude":2.3522}', name='get_weather'), type='function', extra_content={'google': {'thought_signature': 'EuUCCuICAWkUfROo0jINR+T6J+en5vyNasQyfTnMoXMIMD4dHXobzA5QmrDwCQYt6PtzhLmo4S+XJHWwa+xlqbEMQmu7lbXqm/ClrtLwCvttUnD4uf8OSTBHTj9yC7NPcf6hONjnyDl13dpdMhI2vSGVMTlPS8CDFvkqbHlM5mtAmIoBL0jpVyyVDl44c9NbdFwC0fFY3YoGBQ5mRubukA2aTR0M2jHXb4WURRnfI7VXYb6S65bDxAgfNxGrG+DEXBM1ks2+o4nXe+uB0qbvdBDccc6TKH0TjxOc3ZWs9UwdiVG+VhWmrsJvdfA7qqwaYmfvQ2aQiiFP2ui6jeBFxDcWo/G1UaDQ2zjX3lJmTQ02dtY4NBU7dQ0kPGZK+AYK7qbZFMaMLjDfTfiuau3h9YecdrS65yROH1NUEN/LKQzbOcOlL110zPM449j0Oghuvzw3w6RHm4GbiAByp1m2yGNnGiWGMJBW'}})]
Text response: None



## 2. Executing Tool Calls

The model returns a function name and JSON arguments.
Python validates the request and executes the function.

This section tests tool execution offline, without
making a Gemini API request.


In [10]:

def execute_tool(name, argument_json, weather_fn=get_weather):
    if name != "get_weather":
        raise ValueError(f"Unknown tool: {name}")

    arguments = json.loads(argument_json)

    if set(arguments) != {"latitude", "longitude"}:
        raise ValueError("Expected latitude and longitude")

    latitude = arguments["latitude"]
    longitude = arguments["longitude"]

    if type(latitude) not in (int, float) or type(longitude) not in (int, float):
        raise TypeError("Coordinates must be numbers")

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        raise ValueError("Coordinates are out of range")

    temperature = weather_fn(latitude, longitude)

    return {"temperature_celsius": temperature}


# Simulated request from a model
mock_call = {
    "name": "get_weather",
    "arguments": '{"latitude": 48.8566, "longitude": 2.3522}'
}

# Fixed temperature to test without any network request
fake_weather = lambda lat, lon: 13.2

result = execute_tool(
    mock_call["name"],
    mock_call["arguments"],
    weather_fn=fake_weather
)

print("Offline tool result:", result)


Offline tool result: {'temperature_celsius': 13.2}



## 3. Live Manual Function Calling

This example demonstrates the complete function-calling cycle:

1. Gemini requests the weather tool.
2. Python validates and executes the function.
3. The result is added to the conversation history.
4. Gemini generates a final answer using the temperature.

The workflow requires two Gemini requests when a tool is called.

In [15]:


messages = [{
    "role": "user",
    "content": (
        "Use the weather tool to check the current temperature "
        "in Paris (48.8566, 2.3522)."
    )
}]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

assistant_message = first.choices[0].message

if assistant_message.tool_calls:
    messages.append(
        assistant_message.model_dump(exclude_none=True)
    )

    for call in assistant_message.tool_calls:
        result = execute_tool(
            call.function.name,
            call.function.arguments
        )

        print("Tool:", call.function.name)
        print("Result:", result)

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result)
        })

    second = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    print("Final answer:", second.choices[0].message.content)

else:
    print("No tool requested:", assistant_message.content)


Tool: get_weather
Result: {'temperature_celsius': 16.7}
Final answer: The current temperature in Paris is **16.7°C**.



## 4. Agent Loop

An agent loop repeatedly requests model output, executes
any requested tools, and continues until the model
produces a final answer.

The loop includes a maximum-iteration limit to prevent
unbounded execution.

The example below uses simulated model responses.


In [13]:

from types import SimpleNamespace

def agent_loop(initial_messages, request_model, max_iterations=3,
               weather_fn=get_weather):

    history = list(initial_messages)

    for iteration in range(1, max_iterations + 1):
        assistant_message = request_model(history)
        calls = assistant_message.tool_calls or []

        if not calls:
            return assistant_message.content, iteration

        history.append(
            assistant_message.model_dump(exclude_none=True)
        )

        for call in calls:
            result = execute_tool(
                call.function.name,
                call.function.arguments,
                weather_fn=weather_fn
            )

            history.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

    raise RuntimeError("Maximum agent iterations reached")


# Simulated model messages
class MockMessage:
    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls

    def model_dump(self, exclude_none=True):
        return {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments
                    }
                }
                for call in self.tool_calls
            ]
        }


mock_tool = SimpleNamespace(
    id="test-call-1",
    function=SimpleNamespace(
        name="get_weather",
        arguments='{"latitude": 48.8566, "longitude": 2.3522}'
    )
)

mock_responses = iter([
    MockMessage(tool_calls=[mock_tool]),
    MockMessage(content="The sample temperature in Paris is 13.2°C.")
])

answer, iterations = agent_loop(
    [{"role": "user", "content": "What's the weather in Paris?"}],
    request_model=lambda history: next(mock_responses),
    weather_fn=lambda lat, lon: 13.2
)

print("Answer:", answer)
print("Iterations:", iterations)


Answer: The sample temperature in Paris is 13.2°C.
Iterations: 2



## Findings

- Gemini successfully requested `get_weather` with the
  correct coordinates for Paris.
- Python validated and executed the requested function.
- The live workflow returned the weather result to Gemini,
  which generated a final answer.
- Tool-call information and results were preserved in
  conversation history.
- The agent-loop example demonstrated the same execution
  pattern using simulated model responses.
- A maximum-iteration limit prevents an unbounded agent loop.

### Limitations

The agent-loop example uses simulated model responses
rather than a live autonomous Gemini agent. The live
example demonstrates one successful tool-call cycle.

Weather results reflect the time of execution.
